# 1. http-to-bucket

Bajamos el parquet de yellow taxi de enero 2025 (la url la da el pdf del parcial) y lo subimos al bucket `taxis` de Minio. Todo con dlt, y para leer el parquet uso pyarrow como pide el enunciado.

In [1]:
import dlt
import requests
import pyarrow.parquet as pq
import io

Url del archivo, tal cual la del pdf.

In [2]:
PARQUET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"

Este resource descarga el archivo con requests y lo lee con pyarrow (no pandas, porque el parcial pide pyarrow puntualmente). dlt recibe la tabla y ya sabe guardarla como parquet en el destino.

In [3]:
@dlt.resource(name="yellow_tripdata", write_disposition="replace")
def yellow_tripdata():
    response = requests.get(PARQUET_URL)
    response.raise_for_status()
    table = pq.read_table(io.BytesIO(response.content))
    yield table

El pipeline_name (`http_to_bucket`) tiene que coincidir con la sección del secrets.toml, ahí es donde dlt saca las credenciales de Minio. Así no queda nada de credenciales escrito en el notebook.

In [4]:
pipeline = dlt.pipeline(
    pipeline_name="http_to_bucket",
    destination=dlt.destinations.filesystem(bucket_url="s3://taxis"),
    dataset_name="taxis_raw",
)

Corremos pidiendo parquet como formato de carga.

In [5]:
load_info = pipeline.run(
    yellow_tripdata,
    loader_file_format="parquet",
)
print(load_info)

Pipeline http_to_bucket load step finished in 2.25 seconds
1 load package(s) were loaded to destination filesystem and into dataset taxis_raw
The filesystem destination used s3://taxis location to store data
Load package 1788806023.9298635 is LOADED and contains no failed jobs
